In [5]:
!pip install ollama
!pip install fitz
!pip install frontend
!pip install PyMuPDF
!pip install pyreadr
!pip install openpyxl
!pip install qwen_vl_utils
!pip install pillow
!pip install torch
!pip install torchvision
#!pip install tabula-py
#!pip install jpype1
!pip install pdfplumber
#!pip install camelot-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 78.1 MB/s eta 0:00:00a 0:00:01


In [281]:
!pip install pdf2image

In [1]:
%cd ../../daan/Chloe-project/

/daan/Chloe-project


In [6]:
import sys
import base64
import os
import requests
import ollama
import fitz  # PyMuPDF
import io
import re
import pandas as pd 
import numpy as np 
import json
import time
import pyreadr
from openpyxl import Workbook, load_workbook
from pathlib import Path
import importlib
import functions
importlib.reload(functions)


SEED = 1

# Show all columns without truncation
pd.set_option("display.max_colwidth", None)  

# If you also want to see all rows/columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [7]:
import pdfplumber

study_id = "1835" #"2678"

pdf_path = os.path.join(f"extraction_papers/{study_id}.pdf")

extracted_text = ""
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:  # avoid None for empty pages
            extracted_text += text + "\n\n"  # add spacing between pages

In [5]:
print(extracted_text)

Ecotoxicology and Environmental Safety 292 (2025) 117953
Contents lists available at ScienceDirect
Ecotoxicology and Environmental Safety
journal homepage: www.elsevier.com/locate/ecoenv
Unraveling fishers’ perceptions: Impact of mining on fish yield and
diversity in Mwenga, South Kivu, Eastern Democratic Republic of Congo
Dieudonn ´ e Shukuru Wassoa,b,*, Daud Kassama , Adolphe Kwakanaba Mwezec,
Socrate Kamani Tungidib, Emmanuel Tulinabo Ahanyirweb , Christian Masumbuko Barakab,
Rodrigue Balthazar Basengere Ayagirweb,d
aAfrica Centre of Excellence in Aquaculture and Fisheries, Department of Aquaculture Sciences and Fisheries, Lilongwe University of Agriculture and Natural Resources
(LUANAR), Lilongwe P. O. Box 219, Malawi
bDepartment of Animal Sciences, Evangelical University in Africa, Bukavu P.O. Box 3323, the Democratic Republic of the Congo
cDepartment of Environment and Earth Resources Management, Evangelical University in Africa, Bukavu P.O. Box 3323, the Democratic Republic of t

In [433]:
with open(f"training_set/markdown/{study_id}.md", 'r') as f: 
    markdown_text = f.read()

print(markdown_text)

# Unraveling fishers' perceptions: Impact of mining on fish yield and diversity in Mwenga, South Kivu, Eastern Democratic Republic of Congo
Dieudonné Wasso
Daud Kassam
Adolphe Mweze
Socrate Tungidi
Emmanuel Ahanyirwe
Christian Baraka
Rodrigue Ayagirwe

Africa Centre of Excellence in Aquaculture and Fisheries
Department of Aquaculture Sciences and Fisheries
Lilongwe University of Agriculture and Natural Resources (LUANAR)

P. O. Box 219
Lilongwe
Malawi, Department of Animal Sciences
Evangelical University in Africa

P.O. Box 3323
Bukavu
the Democratic Republic of the Congo, Africa Centre of Excellence in Aquaculture and Fisheries
Department of Aquaculture Sciences and Fisheries
Lilongwe University of Agriculture and Natural Resources (LUANAR)

P. O. Box 219
Lilongwe
Malawi, Department of Animal Sciences
Evangelical University in Africa

P.O. Box 3323
Bukavu
the Democratic Republic of the Congo, Department of Animal Sciences
Evangelical University in Africa

P.O. Box 3323
Bukavu
the Demo

In [434]:
# Example usage of pdf_to_images(): 
image_paths = pdf_to_images(study_id)
print(f"Saved {len(image_paths)} images:", image_paths)

Saved 12 images: ['training_set/images/1835_1.png', 'training_set/images/1835_2.png', 'training_set/images/1835_3.png', 'training_set/images/1835_4.png', 'training_set/images/1835_5.png', 'training_set/images/1835_6.png', 'training_set/images/1835_7.png', 'training_set/images/1835_8.png', 'training_set/images/1835_9.png', 'training_set/images/1835_10.png', 'training_set/images/1835_11.png', 'training_set/images/1835_12.png']


In [241]:
# setup ollama client: 

from ollama import Client

ollama = Client(host="http://ollama.runai-shared.svc.cluster.local")

MODEL =  "qwen2.5vl:72b" #"qwen2.5vl:72b" #https://ollama.com/library/qwen2.5vl
MODEL_Lang = "gpt-oss:120b"

In [242]:
#ollama.pull(MODEL)
for progress in ollama.pull(MODEL, stream=True):
    print(progress)

status='pulling manifest' completed=None total=None digest=None
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='sha256:ae3823d63bf19077ec4c484843defaf89c4494a61d32db9b1e38c9bbd2343cb4'
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='sha256:ae3823d63bf19077ec4c484843defaf89c4494a61d32db9b1e38c9bbd2343cb4'
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='sha256:ae3823d63bf19077ec4c484843defaf89c4494a61d32db9b1e38c9bbd2343cb4'
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='sha256:ae3823d63bf19077ec4c484843defaf89c4494a61d32db9b1e38c9bbd2343cb4'
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='sha256:ae3823d63bf19077ec4c484843defaf89c4494a61d32db9b1e38c9bbd2343cb4'
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='sha256:ae3823d63bf19077ec4c484843defaf89c4494a61d32db9b1e38c9bbd2343cb4'
status='pulling ae3823d63bf1' completed=None total=48709030400 digest='s

In [243]:
for progress in ollama.pull(MODEL_Lang, stream=True):
    print(progress)

status='pulling manifest' completed=None total=None digest=None
status='pulling 6be6d66a3f54' completed=None total=65369799840 digest='sha256:6be6d66a3f546d8c19b130dc41dc24b2fc159f84ffbc76a0ee0676205083cf5a'
status='pulling 6be6d66a3f54' completed=None total=65369799840 digest='sha256:6be6d66a3f546d8c19b130dc41dc24b2fc159f84ffbc76a0ee0676205083cf5a'
status='pulling 6be6d66a3f54' completed=None total=65369799840 digest='sha256:6be6d66a3f546d8c19b130dc41dc24b2fc159f84ffbc76a0ee0676205083cf5a'
status='pulling 6be6d66a3f54' completed=None total=65369799840 digest='sha256:6be6d66a3f546d8c19b130dc41dc24b2fc159f84ffbc76a0ee0676205083cf5a'
status='pulling 6be6d66a3f54' completed=None total=65369799840 digest='sha256:6be6d66a3f546d8c19b130dc41dc24b2fc159f84ffbc76a0ee0676205083cf5a'
status='pulling 6be6d66a3f54' completed=601598 total=65369799840 digest='sha256:6be6d66a3f546d8c19b130dc41dc24b2fc159f84ffbc76a0ee0676205083cf5a'
status='pulling 6be6d66a3f54' completed=5938052 total=65369799840 dige

In [244]:
# list models 
for mod in ollama.list().models:
    print(mod)
    print('\n')

model='gpt-oss:120b' modified_at=datetime.datetime(2025, 12, 7, 8, 5, 53, 188877, tzinfo=TzInfo(UTC)) digest='a951a23b46a1f6093dafee2ea481d634b4e31ac720a8a16f3f91e04f5a40ecd9' size=65369818941 details=ModelDetails(parent_model='', format='gguf', family='gptoss', families=['gptoss'], parameter_size='116.8B', quantization_level='MXFP4')


model='qwen2.5vl:72b' modified_at=datetime.datetime(2025, 12, 7, 7, 55, 17, 870409, tzinfo=TzInfo(UTC)) digest='05ea68274581d182f474987ced445c380fccff8c0f9ef8a9bdb48c4ee2ba9620' size=48709042849 details=ModelDetails(parent_model='', format='gguf', family='qwen25vl', families=['qwen25vl'], parameter_size='73.4B', quantization_level='Q4_K_M')




In [272]:
prompt_format = {
    "type": "object",
    "properties": {
        "scale": {
            "type": "string",
            "enum": ["Local", "Regional", "Global"],
            # "description": (
            #     "Local: The paper focuses on a case study from a single mine.\n"
            #     "Regional: This study focuses on a region of the world. This could be subnational (a series of case studies within an area or a region within a country), "
            #     "national (the country level impact from a single country), multinational (an area that exceeds country bounds), or an area in international waters.\n"
            #     "Global: The study is a global level analysis. "
            # ),
        },
        "country_or_continent": {
            "type": "string",
            # "description": "Which country, countries or continent is this study focused?",
        },
        "specific_location": {
            "type": "string",
            # "description": "Where specifically is the study focused? I.e., Central Africa, Clarion Clipperton Zone, Global, etc.",
        },
        "mined_commodities": {
            "type": "string",
            # "description": (
            #     "List all the commodities studied (e.g., mined) in this paper (this could be one or multiple). List any specific commodities "
            #     "that are being mined or were mined in the past. For example: If the text states that the studied mine is a coal mine, then the commodity is coal. "
            #     "Polymetallic nodules in the CCZ contain: Mn, Cu, Co, Ni."
            # ),
        },
        "mining_stage": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Mineral occurrences","Exploration/Pre-operational","Operational","Closed","Rehabilitation","NA"]
            },
            "uniqueItems": True,
            # "description": (
            #     "What is the stage of mining? Multiple options may be applicable. "
            #     "Mineral occurrences: Where a known commodity is located, this could be any stage of mining (exploration, operational, closed, etc.). "
            #     "Exploration/Pre-operational: Where a commodity is known to occur, but mining activities have not commenced. "
            #     "Operational: Mining activities are occurring. The mine is active (e.g., production reported). "
            #     "Closed: Mining activities have concluded, the site could be abandoned or rehabilitated/relinquished. "
            #     "Rehabilitation: Treatment or management of land/water disturbed by mining. Mining stage may also be operational and/or closed."                    
            # ),
            "default":[]
        },
        "type_of_mining_activity": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Open cut","Underground","Mountaintop removal","In-situ leaching","Placer mining","Dredging","Artisanal small-scale mining","Nodule mining","Sulphide mining","Crust mining","Quarrying", "NA"],
            },
            "uniqueItems": True,
            # "description": (
            #     "If this information is not explicitly stated in the study, select NA. Multiple options may be applicable."
            # ),
            "default": []
        },
        "experimental_design": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Mining area vs non-mining area","Current mining vs post-mining","Pre-mining vs current mining","Pre-mining vs post-mining","Pre-mining vs hypothetical mining", "NA"],
            },
            "uniqueItems": True,
            # "description": (
            #     "What comparison is made in mining conditions to determine biodiversity impacts? "
            #     "Mining area vs non-mining area: The impacts of mining in a mining area (any stage) are compared to an untouched area not used for mining. "
            #     "Current mining vs post-mining: The impacts of mining in an area currently being mined are compared to a different area that has undergone mining in the past. "
            #     "Pre-mining vs current mining: The impacts of current mining activities are compared to the state of biodiversity before mining commenced in the same site/area. "
            #     "Pre-mining vs post-mining: The impacts of mining on biodiversity are compared between an area that has undergone mining (after operations have ceased) and the same area before mining occurred. "
            #     "Pre-mining vs hypothetical mining: A mining area (pre-operation) is compared to a hypothetical current/post-mining scenario. "
            #     "Multiple options may be applicable."
            # ),
            "default":[]
        },
        "taxonomic_groups": {
            "type": "string",
            # "description": (
            #     "Which species group/s are investigated in this study? "
            #     "Please be specific, if possible (i.e., salamanders, skinks), otherwise use broader taxonomic group (i.e., mammals, amphibians, etc.)."
            # ),
        },
        "biomes_of_assessment": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Marine", "Freshwater", "Grassland", "Savanna", "Shrubland", "Wetlands", "Rocky areas", "Caves and subterranean", "Forest", "Desert", "Tundra", "NA"]
            },
            "uniqueItems": True,
            # "description": (
            #     "Select the biome in which the biodiversity assessment was conducted (multiple options may be applicable). If this information is not explicitly stated in the study, select NA."
            # ),
            "default": []
        },
        "method_of_assessment": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Spatial modelling","Expert opinion","Statistical analysis","In-situ sampling","Laboratory experiment","Laboratory analysis", "Interviews and questionnaires", "NA"],
            },
            "uniqueItems": True,
            # "description": (
            #     "What method is used to determine the impact or pressure on biodiversity from mining? Multiple options may be applicable. "
            #     "Spatial modelling: A spatial analysis where the mining data is intersected or overlayed with the biodiversity indicator (use of mapping software required for this selection). "
            #     "Expert opinion: The impacts to biodiversity are determined from an expert workshop or from expert judgement. "
            #     "Statistical modelling: Analysis of quantitative data to formulate results. "
            #     "In-situ sampling: Sampling is completed in two or more locations, which can be both inside and outside the impacted area, to compare biodiversity values. "
            #     "Laboratory experiment: Experiments are conducted in a laboratory setting to simulate the impacts of mining on biodiversity. "
            #     "Laboratory analysis: A lab setting is used to assess biodiversity data collected from the field and produce results. "
            #     "Interviews and questionnaires: The biodiversity impacts are determined from questionnaires or interviews with individuals from local communities, through their observations of environmental change."
            # ),
            "default":[]
        },
        "specific_methodology": {
            "type": "string",
            # "description": "Please detail the specific methodology used in the study to quantify the indirect impacts of mining on biodiversity.",
        },
        "temporal_scale": {
            "type": "string",
            "enum": ["1-5", "5-20", "20-50", "50-100", "100-1000", "1000+","NA"],
            # "description": "Over what time period (years) have the impacts to biodiversity been measured? Select 0 if measurements were taken only once. Select NA if no temporal scale is provided.",
        },
        "type_of_impact_or_pressure": {
                "type": "string",
                "enum": ["Pressure","Future impact","Observed impact"],
        },
        "impact_pathway": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ["Land/Seabed use","Water depletion","Ecotoxicity","Acidification","Climate change","Eutrophication","NA"],
            },
            "uniqueItems": True,
            "default": []
        },
        "description_of_impact_or_pressure": {
            "type": "string",
        },
        "spatial_scale": {
            "type": "string",
            "enum": ["0-1", "1-5", "5-10", "10-20", "20-50", "50-100", "100+","NA"],
        }
    },
    "required": [
        "scale","country_or_continent","specific_location","mined_commodities","mining_stage","type_of_mining_activity",
        "experimental_design",
        "taxonomic_groups","biomes_of_assessment","method_of_assessment","specific_methodology","temporal_scale", 
        "type_of_impact_or_pressure", "impact_pathway",
        "description_of_impact_or_pressure","spatial_scale"
    ],
    "additionalProperties": False
}


json_format_mines = {
    "type": "object",
    "properties": {
        "mine_names": {
            "type": "array", 
            "items": {"type": "string"},
            "description": (
                "List of mines that are individually assessed in the study. ",
                "To clarify: Each mine listed must be a single mine for which impacts are estimated and presented individually in the paper. ")
        }
    },
    "required": ["mine_names"],
    "additionalProperties": False
}

prompt_format_metrics = {
    "type": "array",
    "description": "List each metric used in the study to measure the impacts of mining.",
    "items": {
        "type": "object",
        "properties": {
            "metric": {
                "type": "string",
                "description": "Metric used in the study to measure the impacts of mining."
            },
            "metric_category": {
                "type": "string",
                "enum": ["Biodiversity metric","(Sub)-organismal metric","Environmental metric"],
                "description": "Metric category."
                }
        },
        "required": [
            "metric", "metric_category"
        ],
        "additionalProperties": False
    }
}


prompt_format_single_metric = {
    "type": "object",
    "properties": {
        "metric": {
            "type": "string",
            "description": "Metric used in the study to measure the impacts of mining."
        },
        "impact_direction": {
            "type": "string",
            "enum": ["No Impact","Increased","Decreased","Changed","Variable","NA"],
            "description": (
                "No Impact: Relative to a reference, mining has caused neither a gain nor loss to biodiversity values (For rehabilitation studies: rehabilitation has increased biodiversity values to be equivalent to reference site/s or pre-mining conditions). "
                "Increased: Relative to a reference, mining is associated with an increase in the metric (For rehabilitation studies: rehabilitation has increased biodiversity values above reference site/s or pre-mining conditions). "
                "Decreased: Relative to a reference, mining is associated with a decrease in the metric (For rehabilitation studies: 1) rehabilitation has increased biodiversity values but not above reference site/s or pre-mining conditions. 2) Rehab has decreased biodiversity values or is having no positive impact). "
                "Changed: The impacts of this metric cannot be measured as increased or decreased but there has been a change when comparing between different conditions. "
                "Variable: The impacts are reported as both increasing biodiversity values in some respects and decreasing them in others. For example, if the paper assesses different species individually and the direction is increased for one species and decreased for another."
            ),
        },
        "significance": {
            "type": "string",
            "enum": ["Significant","Non-significant", "NA"],
            "description": (
                "Is the change in metric stated to be significant (p value < 0.05) or non-significant? Choose NA if this is not stated in the text."
            ),
        },
        "p_value": {
            "type": "string",
            "description": (
                "What is the p-value associated with the metric change? Choose 'NA' if this is not reported in the text."
            ),
        },
        "effect_size": {
            "type": "string",
            "description": (
                "What is the effect size associated with the metric change? Choose 'NA' if this is not reported in the text."
            ),
        }
    },
    "required": [
        "metric", "impact_direction", "significance", "p_value", "effect_size"
    ],
    "additionalProperties": False
}


In [273]:
with open("system_prompt.txt", newline=None) as f:
    system_prompt = f.read()

with open("text_prompt.txt", newline=None) as f:
    text_prompt = f.read()

with open("text_prompt_mine.txt", newline=None) as f:
    text_prompt_mine = f.read()
    
with open("text_prompt_diversity_metrics.txt", newline=None) as f:
    text_prompt_diversity_metrics = f.read()

with open("text_prompt_diversity_metrics_2.txt", newline=None) as f:
    text_prompt_diversity_metrics_2 = f.read()

import importlib
import functions
importlib.reload(functions)

<module 'functions' from '/daan/Chloe-project/functions.py'>

In [72]:
df_full['Study ID'] = df_full['Study ID'].astype(str)
df_test = df_full[df_full['Study ID'].isin(study_test)]
df_test.to_excel("output-test.xlsx", index=False)

In [ ]:
# ----------------------------------------------------------------
# Excel setup

all_cols = ["Scale", "Country/Continent", "Specific location", "Mined commodities", "Mining stage", "Type of mining activity", "Experimental design",
"Taxonomic groups", "Biomes of assessment","Method of assessment","Specific methodology", "Temporal scale", "Type of impact or pressure", 
"Impact pathway", "Description of impact or pressure", "Spatial scale", "Metric", "Metric category", "Impact direction", "Significance", 
"P value", "Effect size", "Response study", "Response metric", "Extracted tables", "Study ID"]

output_file = Path("output-rest_v4.xlsx")
if output_file.exists():
    wb = load_workbook(output_file)
    ws = wb.active
else:
    wb = Workbook()
    ws = wb.active
    ws.append(list(all_cols))
    wb.save(output_file)

In [ ]:
study_train = ["2544","1825","1835","1986","2063","2067","2546","2678","2714","2758"]
study_test = ["1879", "1923", "2032", "2105", "2135", "2298", "2570", "2601", "2643", "2651", "2754", "2846", "2890", "3197", "3436"]
study_rest = ["1829", "1843", "1854", "1859", "1874", "1884", "1885", "1894", "1898", "1926", "1927", "1952", "1960", "1969", "2010", "2016", "2017", "2028", "2049", "2051", "2084", "2095", "2100", "2127", "2139", "2153", "2157", "2165", "2185", "2193", "2229", "2232", "2243", "2244", "2245", "2261", "2271", "2279", "2285", "2289", "2296", "2307", "2308", "2320", "2325", "2327", "2332", "2335", "2337", "2370", "2378", "2381", "2404", "2410", "2433", "2447", "2448", "2452", "2471", "2472", "2483", "2507", "2524", "2530", "2531", "2541", "2550", "2569", "2574", "2581", "2595", "2598", "2602", "2607", "2614", "2616", "2617", "2618", "2621", "2622", "2636", "2637", "2645", "2646", "2649", "2652", "2659", "2668", "2673", "2680", "2681", "2691", "2696", "2703", "2711", "2721", "2724", "2727", "2750", "2760", "2787", "2793", "2823", "2841", "2858", "2869", "2883", "2895", "2909", "2932", "2970", "2997", "3033", "3069", "3082", "3085", "3133", "3170", "3174", "3176", "3247", "3262", "3288"]

len(study_train), len(study_test), len(study_rest)

In [ ]:
import sys
from datetime import datetime

class Logger(object):
    def __init__(self, filename="log.txt"):
        self.terminal = sys.stdout
        self.log = open(filename, "a", encoding="utf-8")

    def write(self, message):
        if message.strip():
            # Define timestamp each time a message is written
            timestamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ")
            message = "".join([timestamp + line for line in message.splitlines(True)])
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        pass  # needed for compatibility with sys.stdout

# Redirect print() output
sys.stdout = Logger("log-rest.txt")

In [ ]:
import importlib
import functions
importlib.reload(functions)

# ----------------------------------------------------------------
# Loop through abstracts
start_time = time.time()

i = 0
for study_id in study_rest:  
    print(f"Iteration: {i}\t\t Study: {study_id}")

    image_paths = functions.pdf_to_images(study_id)
    
    extracted_text = ""
    with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:  # avoid None for empty pages
                extracted_text += text + "\n\n"  # add spacing between pages

    # markdown text extracted with the GROBID model
    with open(f"extraction_papers/{study_id}.md", 'r') as f: 
       markdown_text = f.read()

    # 1. extract tables from the text, to ensure correct interpretation of results 
    print("Extracting tables from the text...")
    text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 
    
    # 2. get names of individual mines, if assessed individually in the study: 
    print("Determining whether multiple mines are assessed individually in the study...") 
    mines_prompt = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text" + extracted_text + "\n\nTables from the text: " + text_tables + """\n\n
    If this mining-impact study individually assessed the impacts of mining of one or more single mines, list each individual mine. 
    Only list single, individual mines, don't list different study locations that assessed the same mine. To clarify: Each listed mine must be one single mine for which 
    impacts are estimated and presented individually in the paper.
    """

    mines, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, mines_prompt, json_format_mines, image_paths) 
    #mines_response, _ = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = mines_prompt)
    #mines = functions.get_json_table(ollama, MODEL_Lang, SEED, mines_response, json_format_mines)
    
    if (mines is None) or (mines=={}) or (mines.get("mine_names", [])==[]) or (len(mines.get("mine_names", []))==1): 
        print("No (multiple) individual mines found, starting extraction for complete study...")
        
        df = functions.get_full_response_json(ollama, 
                                              MODEL, 
                                              MODEL_Lang, 
                                              SEED, 
                                              study_id, 
                                              system_prompt, 
                                              text_prompt, 
                                              text_prompt_diversity_metrics, 
                                              text_prompt_diversity_metrics_2, 
                                              text_tables, 
                                              prompt_format, 
                                              prompt_format_metrics, 
                                              prompt_format_single_metric, 
                                              image_paths, 
                                              extracted_text,
                                              markdown_text
                                       )

        df = df[all_cols] # reorder columns 

        df["P value"] = (
            df["P value"]
            .astype(str)
            .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
        )
    
        # append to excel workbook 
        for j, row in enumerate(df.itertuples(index=False), start=1):
            ws.append(row)
            
        print("Done.")
        
    else: 
        print(f"Multiple mines found: {', '.join(mines.get("mine_names"))}")

        for mine in mines.get("mine_names"):
            print(f"Starting extraction for mine: {mine}...")
            
            df = functions.get_full_response_json(ollama, 
                                                  MODEL, 
                                                  MODEL_Lang, 
                                                  SEED, 
                                                  study_id, 
                                                  system_prompt, 
                                                  text_prompt_mine.format(mine=mine), 
                                                  text_prompt_diversity_metrics,
                                                  text_prompt_diversity_metrics_2, 
                                                  text_tables, 
                                                  prompt_format, 
                                                  prompt_format_metrics, 
                                                  prompt_format_single_metric, 
                                                  image_paths, 
                                                  extracted_text,
                                                  markdown_text,
                                                  prompt_prefix = f"For the mine named '{mine}' specifically, do the following:"
                                                 )

            df['Scale'] = "Local" # per definition 
            df = df[all_cols] # reorder columns 

            df["P value"] = (
                df["P value"]
                .astype(str)
                .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
            )
    
            # append to excel workbook 
            for j, row in enumerate(df.itertuples(index=False), start=1):
                ws.append(row)
                
            print("Done.")
            
    # make sure all elements are strings 
    #json_table = json_table.map(lambda x: ", ".join(x) if isinstance(x, list) else x)

    wb.save(output_file)
    i+=1

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")

In [ ]:
# Restore normal printing
sys.stdout.log.close()
sys.stdout = sys.stdout.terminal

# Restore the original stdout so print() shows in the notebook again
if hasattr(sys.stdout, "terminal"):
    sys.stdout = sys.stdout.terminal

In [40]:
# Finished in 106.99 minutes

In [ ]:
# ----------------------------------------------------------------
# Excel setup

all_cols = ["Scale", "Country/Continent", "Specific location", "Mined commodities", "Mining stage", "Type of mining activity", "Experimental design",
"Taxonomic groups", "Biomes of assessment","Method of assessment","Specific methodology", "Temporal scale", "Type of impact or pressure", 
"Impact pathway", "Description of impact or pressure", "Spatial scale", "Metric", "Metric category", "Impact direction", "Significance", 
"P value", "Effect size", "Response study", "Response metric", "Extracted tables", "Study ID"]

output_file = Path("output-test_v4.xlsx")
if output_file.exists():
    wb = load_workbook(output_file)
    ws = wb.active
else:
    wb = Workbook()
    ws = wb.active
    ws.append(list(all_cols))
    wb.save(output_file)

# Redirect print() output
sys.stdout = Logger("log-test.txt")

import importlib
import functions
importlib.reload(functions)

# ----------------------------------------------------------------
# Loop through abstracts
start_time = time.time()

i = 0
for study_id in study_test:  
    print(f"Iteration: {i}\t\t Study: {study_id}")

    image_paths = functions.pdf_to_images(study_id)
    
    extracted_text = ""
    with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:  # avoid None for empty pages
                extracted_text += text + "\n\n"  # add spacing between pages

    # markdown text extracted with the GROBID model
    with open(f"extraction_papers/{study_id}.md", 'r') as f: 
       markdown_text = f.read()

    # 1. extract tables from the text, to ensure correct interpretation of results 
    print("Extracting tables from the text...")
    text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 
    
    # 2. get names of individual mines, if assessed individually in the study: 
    print("Determining whether multiple mines are assessed individually in the study...") 
    mines_prompt = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text" + extracted_text + "\n\nTables from the text: " + text_tables + """\n\n
    If this mining-impact study individually assessed the impacts of mining of one or more single mines, list each individual mine. 
    Only list single, individual mines, don't list different study locations that assessed the same mine. To clarify: Each listed mine must be one single mine for which 
    impacts are estimated and presented individually in the paper.
    """

    mines, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, mines_prompt, json_format_mines, image_paths) 
    #mines_response, _ = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = mines_prompt)
    #mines = functions.get_json_table(ollama, MODEL_Lang, SEED, mines_response, json_format_mines)
    
    if (mines is None) or (mines=={}) or (mines.get("mine_names", [])==[]) or (len(mines.get("mine_names", []))==1): 
        print("No (multiple) individual mines found, starting extraction for complete study...")
        
        df = functions.get_full_response_json(ollama, 
                                              MODEL, 
                                              MODEL_Lang, 
                                              SEED, 
                                              study_id, 
                                              system_prompt, 
                                              text_prompt, 
                                              text_prompt_diversity_metrics, 
                                              text_prompt_diversity_metrics_2, 
                                              text_tables, 
                                              prompt_format, 
                                              prompt_format_metrics, 
                                              prompt_format_single_metric, 
                                              image_paths, 
                                              extracted_text,
                                              markdown_text
                                       )

        df = df[all_cols] # reorder columns 

        df["P value"] = (
            df["P value"]
            .astype(str)
            .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
        )
    
        # append to excel workbook 
        for j, row in enumerate(df.itertuples(index=False), start=1):
            ws.append(row)
            
        print("Done.")
        
    else: 
        print(f"Multiple mines found: {', '.join(mines.get("mine_names"))}")

        for mine in mines.get("mine_names"):
            print(f"Starting extraction for mine: {mine}...")
            
            df = functions.get_full_response_json(ollama, 
                                                  MODEL, 
                                                  MODEL_Lang, 
                                                  SEED, 
                                                  study_id, 
                                                  system_prompt, 
                                                  text_prompt_mine.format(mine=mine), 
                                                  text_prompt_diversity_metrics,
                                                  text_prompt_diversity_metrics_2, 
                                                  text_tables, 
                                                  prompt_format, 
                                                  prompt_format_metrics, 
                                                  prompt_format_single_metric, 
                                                  image_paths, 
                                                  extracted_text,
                                                  markdown_text,
                                                  prompt_prefix = f"For the mine named '{mine}' specifically, do the following:"
                                                 )

            df['Scale'] = "Local" # per definition 
            df = df[all_cols] # reorder columns 

            df["P value"] = (
                df["P value"]
                .astype(str)
                .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
            )
    
            # append to excel workbook 
            for j, row in enumerate(df.itertuples(index=False), start=1):
                ws.append(row)
                
            print("Done.")
            
    # make sure all elements are strings 
    #json_table = json_table.map(lambda x: ", ".join(x) if isinstance(x, list) else x)

    wb.save(output_file)
    i+=1

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")

# Restore normal printing
sys.stdout.log.close()
sys.stdout = sys.stdout.terminal

# Restore the original stdout so print() shows in the notebook again
if hasattr(sys.stdout, "terminal"):
    sys.stdout = sys.stdout.terminal

In [186]:
import importlib
import functions
importlib.reload(functions)

<module 'functions' from '/daan/Chloe-project/functions.py'>

In [278]:
study_id = "2546"

print(f"Iteration: {i}\t\t Study: {study_id}")

image_paths = functions.pdf_to_images(study_id)

extracted_text = ""
with pdfplumber.open(f"extraction_papers/{study_id}.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:  # avoid None for empty pages
            extracted_text += text + "\n\n"  # add spacing between pages

# markdown text extracted with the GROBID model
with open(f"extraction_papers/{study_id}.md", 'r') as f: 
    markdown_text = f.read()

# 1. extract tables from the text, to ensure correct interpretation of results 
#print("Extracting tables from the text...")
#text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 


Iteration: 2		 Study: 2546


In [188]:
# 2. get text response for the metrics (supply images, text and extracted tables): 
user_prompt_2 = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text: " + extracted_text + "\n\nTables from the text: " + text_tables + "\n\n" + text_prompt_diversity_metrics

response_metrics, messages_metrics = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = user_prompt_2)    

if response_metrics is None:
    print("Step 2. is None. Trying again...")
    response_metrics, messages_metrics = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = user_prompt_2)  
    if response_metrics is None: # probable API error or failure to generate 
        print("Failed.") 

json_metrics = functions.get_json_table(ollama, MODEL_Lang, SEED, response_metrics, prompt_format_metrics)
json_metrics

```json
{
  "metrics": [
    {
      "name": "Sediment arsenic concentration (mg/kg)",
      "category": "Environmental metric"
    },
    {
      "name": "Sediment cadmium concentration (mg/kg)",
      "category": "Environmental metric"
    },
    {
      "name": "Sediment lead concentration (mg/kg)",
      "category": "Environmental metric"
    },
    {
      "name": "Sediment zinc concentration (mg/kg)",
      "category": "Environmental metric"
    },
    {
      "name": "Sediment manganese concentration (mg/kg, <0.01)",
      "category": "Environmental metric"
    },
    {
      "name": "Sediment mercury concentration (mg/kg, <0.01)",
      "category": "Environmental metric"
    },
    {
      "name": "Sediment nickel concentration (mg/kg, <0.01)",
      "category": "Environmental metric"
    },
    {
      "name": "Water temperature (°C)",
      "category": "Environmental metric"
    },
    {
      "name": "Water pH",
      "category": "Environmental metric"
    },
    {
      "na

[{'metric': 'Sediment arsenic concentration (mg/kg)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Sediment cadmium concentration (mg/kg)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Sediment lead concentration (mg/kg)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Sediment zinc concentration (mg/kg)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Sediment manganese concentration (mg/kg, <0.01)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Sediment mercury concentration (mg/kg, <0.01)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Sediment nickel concentration (mg/kg, <0.01)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Water temperature (°C)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Water pH', 'metric_category': 'Environmental metric'},
 {'metric': 'Dissolved oxygen (mg/L)',
  'metric_category': 'Environmental metric'},
 {'metric': 'Conductivity (µS or lS)',
  'metri

In [279]:
# 2. get names of individual mines, if assessed individually in the study: 
print("Determining whether multiple mines are assessed individually in the study...") 
mines_prompt = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nRaw text" + extracted_text + "\n\nTables from the text: " + text_tables + """\n\n
    If this mining-impact study assessed the impacts of mining of one or more mines individually, list each individual mine. 
    Only list individual mines, don't list study locations that assessed the same mine. To clarify: These are mines for which 
    impacts are estimated and presented individually in the paper. 
    """

mines, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, mines_prompt, json_format_mines, image_paths) 
    
print(mines)


Determining whether multiple mines are assessed individually in the study...
{'mine_names': []}


In [180]:
if response_metrics is not None: 
        
    cols = ["Metric", "Metric category", "Impact direction", "Significance", "P value", "Effect size", "Response metric"]
    df = pd.DataFrame([[""] * len(cols)],columns=cols) # empty dataframe
    for metric in response_metrics: 
        df_row = pd.DataFrame([[""] * len(cols)],columns=cols) # empty dataframe
        df_row['Metric'] = metric.get('metric', "")
        df_row['Metric category'] = metric.get('metric_category', "")
        df = pd.concat([df, df_row], ignore_index=True)

    df = df[1:]
    
    for m in df['Metric']: 
        print(f"Extracting information for metric: {m}")
        # setup prompt for metric
        user_prompt_3 = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nTables from the text: " + text_tables + "\n\nRaw text" + extracted_text + "\n\n" + f"For the metric {m} specifically, determine:\n" + text_prompt_diversity_metrics_2
        # extract json 
        #response_metric, _ = functions.extract_information_formatted(ollama, MODEL, SEED, system_prompt, user_prompt_3, prompt_format_single_metric, image_paths)
        
        response, messages = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = user_prompt_3)
        print(response)
        response_metric = functions.get_json_table(ollama, MODEL_Lang, SEED, response, prompt_format_single_metric)
        print(response_metric)
        
        # add to df 
        df.loc[df['Metric'] == m, 'Impact direction'] = response_metric.get("impact_direction", "")
        df.loc[df['Metric'] == m, 'Significance']     = response_metric.get("significance", "")
        df.loc[df['Metric'] == m, 'P value']          = response_metric.get("p_value", "")
        df.loc[df['Metric'] == m, 'Effect size']      = response_metric.get("effect_size", "")
        df.loc[df['Metric'] == m, 'Response metric']  = response # save text response
        


Extracting information for metric: Shannon H
**Shannon H (diversity index)**  

| Item | Answer |
|------|--------|
| **a) Impact direction** | **Decreased** – the study reports that higher cadmium and zinc concentrations are associated with lower Shannon H values (i.e., reduced diversity). |
| **b) Significance** | **significant** – the relationship is reported as statistically significant. |
| **c) P‑value** | **0.017** – given for the MANOVA that includes Shannon H (Wilks’ Λ, F₈,₄₈ = 2.65, P = 0.017). |
| **d) Effect size** | **NA** – no explicit effect‑size metric (e.g., Cohen’s d, η²) is provided for Shannon H. |
{'metric': 'Shannon H', 'impact_direction': 'Decreased', 'significance': 'Significant', 'p_value': '0.017', 'effect_size': 'NA'}
Extracting information for metric: Richness J
**Richness J**

| Item | Answer |
|------|--------|
| a) Impact direction | NA |
| b) Significance | NA |
| c) P‑value | NA |
| d) Effect size | NA |

*Explanation*: The manuscript reports significan

In [182]:
df

,Metric,Metric category,Impact direction,Significance,P value,Effect size
1,Shannon H,Biodiversity metric,Decreased,Significant,0.017,NA
2,Richness J,Biodiversity metric,NA,NA,NA,NA
3,Evenness E,Biodiversity metric,NA,NA,NA,NA
4,Percent of Ephemeroptera,Biodiversity metric,NA,NA,NA,NA
5,Percent of Plecoptera,Biodiversity metric,Decreased,Significant,0.017,NA
6,Percent of Trichoptera,Biodiversity metric,NA,NA,NA,NA
7,Total number of collected animals,Biodiversity metric,Decreased,Significant,Cadmium: 0.017; Zinc: 0.014,NA
8,Number of families,Biodiversity metric,Decreased,Significant,0.017,NA
9,Numbers of orders,Biodiversity metric,NA,NA,NA,NA
10,Arsenic concentration,Environmental metric,NA,Non-significant,NA,NA


In [510]:
user_prompt_1 = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\n" + text_prompt
    
response, messages = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, prompt = user_prompt_1)
print("response", response)

response **1. Scale**  
- **Answer:** **Regional**  
- **Reasoning:** The study is confined to the Idaho Panhandle National Forests (IPNF) in northern Idaho, covering 10 streams within a ~5 km² area. This is a sub‑national (regional) assessment rather than a single‑site case study or a worldwide analysis.  

---

**2. Country / Continent**  
- **Answer:** United States (North America)  

---

**3. Specific location**  
- **Answer:** Northern Idaho, USA – Idaho Panhandle National Forests, Mt. Coeur d’Alene area; streams that are tributaries to the lower South Fork of the Coeur d’Alene River.  

---

**4. Mined commodities**  
- **Answer:**  
  1. **Silver** (the historic “silver (galena) mining” mentioned)  
  2. **Lead** (galena is a lead‑sulfide ore, often mined for its lead content)  

*The paper also discusses the presence of cadmium, zinc and arsenic in the waste, but these are treated as contaminants rather than the primary mined commodities.*  

---

**5. Mining stage**  
- **Ans

In [514]:
user_prompt_2 = "Carefully read this mining-impact study:\n\n" + markdown_text + "\n\nTables from the text: " + text_tables + "\n\n" + text_prompt_diversity_metrics
    
response_metrics, messages_metrics = functions.extract_information(ollama, MODEL_Lang, SEED, system_prompt, images=False, image_paths=image_paths, prompt = user_prompt_2)
print("response_metrics", response_metrics)  

response_metrics **A.  Metrics that were actually measured, modelled or otherwise estimated in the paper**

| # | Metric (as written in the article) |
|---|------------------------------------|
| 1 | Shannon **H** diversity index |
| 2 | Shannon **J** (Pielou’s evenness) |
| 3 | Shannon **E** (evenness) |
| 4 | **Richness** value (species‑richness) |
| 5 | **Evenness** value |
| 6 | % **Ephemeroptera** (percent of total organisms) |
| 7 | % **Plecoptera** (percent of total organisms) |
| 8 | % **Trichoptera** (percent of total organisms) |
| 9 | **Total number of collected animals** (abundance) |
|10 | **Number of families** (taxonomic richness at family level) |
|11 | **Number of orders** (taxonomic richness at order level) |
|12 | **Arsenic** concentration in stream sediments (mg kg⁻¹) |
|13 | **Cadmium** concentration in stream sediments (mg kg⁻¹) |
|14 | **Lead** concentration in stream sediments (mg kg⁻¹) |
|15 | **Zinc** concentration in stream sediments (mg kg⁻¹) |
|16 | **Water

In [512]:
json_response = functions.get_json_table(ollama, MODEL_Lang, SEED, system_prompt, user_prompt_1, response, prompt_format)
print(json_response)

{'scale': 'Regional', 'country_or_continent': 'United States (North America)', 'specific_location': "Northern Idaho, Idaho Panhandle National Forests (Mt. Coeur d'Alene area, tributary streams to the South Fork of the Coeur d'Alene River)", 'mined_commodities': 'Silver (galena), Lead, Zinc', 'mining_stage': ['Closed'], 'type_of_mining_activity': ['NA'], 'experimental_design': ['NA'], 'taxonomic_groups': 'Aquatic macroinvertebrate insects (larval stream insects – Ephemeroptera, Plecoptera, Trichoptera)', 'biomes_of_assessment': ['Freshwater'], 'method_of_assessment': ['In-situ sampling', 'Laboratory analysis', 'Statistical analysis'], 'specific_methodology': 'Surber sampler (500‑µm mesh) was used to collect benthic insects from riffle habitats at three sites per stream (30 sites total). Samples were preserved in ethanol, sorted to genus, and counted. Sediment (100\u202fcc) from each riffle was collected, dried, digested (EPA SW846 method 3050) and analyzed for As, Cd, Pb, Zn by ICP‑OES 

In [515]:
json_response_metrics = functions.get_json_table(ollama, MODEL_Lang, SEED, system_prompt, user_prompt_2, response_metrics, prompt_format_metrics)
print(json_response_metrics)

[{'metric': 'Shannon H', 'metric_category': 'Biodiversity metric', 'impact_direction': 'Decreased', 'significance': 'Significant', 'p_value': '.017 (cadmium) / .014 (zinc)', 'effect_size': 'NA'}, {'metric': 'Shannon J', 'metric_category': 'Biodiversity metric', 'impact_direction': 'NA', 'significance': 'NA', 'p_value': 'NA', 'effect_size': 'NA'}, {'metric': 'Shannon E', 'metric_category': 'Biodiversity metric', 'impact_direction': 'NA', 'significance': 'NA', 'p_value': 'NA', 'effect_size': 'NA'}, {'metric': '% Ephemeroptera', 'metric_category': 'Biodiversity metric', 'impact_direction': 'NA', 'significance': 'NA', 'p_value': 'NA', 'effect_size': 'NA'}, {'metric': '% Plecoptera', 'metric_category': 'Biodiversity metric', 'impact_direction': 'Decreased', 'significance': 'Significant', 'p_value': '.017 (cadmium) / .014 (zinc)', 'effect_size': 'NA'}, {'metric': '% Trichoptera', 'metric_category': 'Biodiversity metric', 'impact_direction': 'NA', 'significance': 'NA', 'p_value': 'NA', 'effec

In [446]:
import importlib
import functions
importlib.reload(functions)

# testing... 

study_id = "2758"

# ----------------------------------------------------------------
# Excel setup

all_cols = ["Scale", "Country/Continent", "Specific location", "Mined commodities", "Mining stage", "Type of mining activity", "Experimental design",
"Taxonomic groups", "Biomes of assessment","Method of assessment","Specific methodology", "Temporal scale", "Type of impact or pressure", 
"Impact pathway", "Description of impact or pressure", "Spatial scale", "Metric", "Metric category", "Impact direction", "Significance", 
"P-value", "Response study", "Response metrics", "Extracted tables", "Study ID"]

output_file = Path(f"output-train_{study_id}.xlsx")
if output_file.exists():
    wb = load_workbook(output_file)
    ws = wb.active
else:
    wb = Workbook()
    ws = wb.active
    ws.append(list(all_cols))
    wb.save(output_file)

# extraction... 
start_time = time.time()

image_paths = pdf_to_images(study_id)

extracted_text = ""
with pdfplumber.open(f"training_set/{study_id}.pdf") as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:  # avoid None for empty pages
            extracted_text += text + "\n\n"  # add spacing between pages

# 1. extract tables from the text, to ensure correct interpretation of results 
print("Extracting tables from the text...")
text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 

# 2. get names of individual mines, if assessed individually in the study: 
print("Determining whether multiple mines are assessed individually in the study...") 
mines, _ = functions.get_mines(ollama, MODEL, SEED, system_prompt, json_format_mines, image_paths) 

if (mines is None) or (mines=={}) or (mines.get("Mine names", [])==[]): 
    print("No individual mines found, starting extraction for complete study...")
    
    df = functions.get_full_response_json(ollama, MODEL, SEED, study_id, system_prompt, text_prompt, text_prompt_diversity_metrics, 
                                    text_tables, prompt_format, prompt_format_metrics, image_paths, extracted_text
                                   )

    df = df[all_cols] # reorder columns 

    # append to excel workbook 
    for j, row in enumerate(df.itertuples(index=False), start=1):
        ws.append(row)
        
    print("Done.")
    
else: 
    print(f"Multiple mines found: {', '.join(mines.get("Mine names"))}")

    for mine in mines.get("Mine names"):
        print(f"Starting extraction for mine: {mine}...")
        
        df = functions.get_full_response_json(ollama, MODEL, SEED, study_id, system_prompt, text_prompt_mine.format(mine=mine), text_prompt_diversity_metrics, 
                                              text_tables, prompt_format, prompt_format_metrics, image_paths, extracted_text,
                                              prompt_prefix = f"For the mine named '{mine}' specifically, do the following:"
                                             )

        df['Scale'] = "Local" # per definition 
        df = df[all_cols] # reorder columns 

        df["P-value"] = (
            df["P-value"]
            .astype(str)
            .apply(lambda x: re.sub(r"^/(\d)", r"\1", x.strip()))  # remove leading "/" before digits
        )

        # append to excel workbook 
        for j, row in enumerate(df.itertuples(index=False), start=1):
            ws.append(row)
            
        print("Done.")

wb.save(output_file)

elapsed = time.time() - start_time
print(f"Finished in {elapsed/60:.2f} minutes")

KeyboardInterrupt: 

In [444]:
# 1. extract tables from the text, to ensure correct interpretation of results 
print("Extracting tables from the text...")
text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 

Extracting tables from the text...


In [445]:
text_tables

"Table 1\n\nWater quality parameters of the Zalya-Elila River (stretching along Mwenga territory).\n\n| Water parameters | Sites surveyed | Average | P-value | Acceptable levels* |\n|------------------|----------------|---------|--------|--------------------|\n|                  | Site A         | Site B  | Site C |                    |\n| Temperature (°C) | 24.45±0.05     | 24.35±0.25 | 24.50±0.10 | 24.43±0.13 | 0.805 | 20–25°C |\n| pH               | 6.25±0.45      | 6.4±0.03  | 5.90±0.20 | 6.18±0.23 | 0.521 | 6.5–8.5 |\n| EC (μS/cm)       | 300.00±90.00b  | 490.00±120.00ab | 845.00±25.00a | 545.00±78.33 | 0.048* | 500–1000 |\n| TDS (mg/L)       | 140.00±40.00b  | 235.00±55.00ab | 410.00±10.00a | 261.67±35.00 | 0.037* | 500–1500 |\n| DO (mg/L)        | 5.21±0.71      | 4.90±0.51 | 7.55±0.20 | 5.89±0.47 | 0.064 | >5.00 |\n| BOD5 (mg/L)      | 3.88±0.82      | 2.81±0.05 | 2.40±0.97 | 3.03±0.61 | 0.441 | <5.00 |\n| COD (mg/L)       | 6.55±4.95      | 6.70±4.50 | 2.45±0.35 | 5.23±3.27 | 

In [449]:
import importlib
import functions
importlib.reload(functions)

messages = [{
            "role": "system",
            "content": system_prompt
    }]
    
messages.append({
        "role": "user",
        "content": "Read this mining-impact study carefully: " + markdown_text + "\n\n" + text_prompt
})

stream = ollama.chat(
    model=MODEL_Lang,
    messages=messages,
    stream=True,
    options={"seed":seed,
             #"num_ctx":4096*10,
             #"num_predict":4096*5,
             #"timeout":3600,
             "temperature":0
            },
)

try:
    # Stream the chunks as they arrive
    response = ""
    for chunk in stream:
        # Each chunk is a dict; content lives here:
        if "message" in chunk and "content" in chunk["message"]:
            text = chunk["message"]["content"]
            #print(text, end="", flush=True)   
            response += text    
except Exception as e:
    print(f"Error calling Ollama API: {e}.")
    response = None

# Append assistant response to history
messages.append({
    "role": "assistant",
    "content": response,
})


print("response", response)
        
# # 2. get text response for the metrics (supply extracted tables): 
# response_metrics, messages_metrics = functions.extract_information(ollama, MODEL, SEED, system_prompt, image_paths,
#                                                          prompt = text_prompt_diversity_metrics + "\n\nTables from the text: " + text_tables
#                                                         )

# print("response_metrics", response_metrics)

response **1. Scale**  
**Answer:** **Local**  
**Reasoning:** The paper investigates the impacts of artisanal small‑scale gold mining on fish biodiversity and water quality in a single territory (Mweso, North Kivu, DRC). It does not aggregate data across multiple regions or countries, nor does it model impacts at a broader scale. Hence it is a case‑study at the local level.

---

**2. Country / Continent**  
**Answer:** **Democratic Republic of the Congo (Africa)**  
**Reasoning:** All field work, sampling, and questionnaires were carried out in the Mweso territory of North Kivu province, which is located in the Democratic Republic of the Congo (DRC) on the African continent.

---

**3. Specific location**  
**Answer:** **Central/Eastern Africa – North Kivu province, Mweso territory (DRC)**  
**Reasoning:** The study repeatedly refers to “Mweso territory, North Kivu, DRC” and to the three sampling sites (Mugunga, Nyamuragira, Lukula) that lie within this sub‑regional area.

---

**4. 

In [448]:
text_prompt

"Thoroughly analyse the attached mining-impact study and create a section for each of the instructions below. Let's think through each task step by step.\n\n\n1. Scale: The scale at which the study is conducted (choose from: local, regional, global). \n- Local: The paper focuses on a case study from a single mine. \n- Regional: This study focuses on a region of the world. This could be subnational (a series of case studies within an area or a region within a country), national (the country level impact from a single country), multinational (an area that exceeds country bounds), or an area in international waters.\n- Global: The study is a global level analysis. \n\n2. Country/Continent: Which country, countries or continent is this study focused?\n\n3. Specific location: Where specifically is the study focused? I.e., Central Africa, Clarion Clipperton Zone, Global, etc.\n\n4. Mined commodities: List all the commodities mined or studied in the study. These can be commodities that are ac

In [240]:
# 1. extract tables from the text, to ensure correct interpretation of results 
print("Extracting tables from the text...")
text_tables, _ = functions.extract_tables(ollama, MODEL, SEED, system_prompt, extracted_text, image_paths) 

Extracting tables from the text...


In [248]:
df

,Scale,Country/Continent,Specific location,Mined commodities,Mining stage,Type of mining activity,Experimental design,Taxonomic groups,Biomes of assessment,Method of assessment,Specific methodology,Temporal scale,Type of impact or pressure,Impact pathway,Description of impact or pressure,Spatial scale,Metric,Metric category,Impact direction,Significance,P-value,Response study,Response metrics,Extracted tables,Study ID


In [251]:
print(text_tables)

TABLE 1. Average Physicochemical Measures of Sampling Sites within the Finniss River in (a) 1992 and (b) 1995

| site | dist., km | depth, m | pH | cond., mS/cm | turb., NTU | temp., °C | D.O. % sat. |
|------|-----------|----------|----|--------------|------------|-----------|-------------|
| (a) 1992 | | | | | | | |
| 6 | -18 | 1.7 | 7.1 | 0.11 | 4.2 | 21.4 | 50 |
| 5 | -0.1 | 1.4 | 6.9 | 0.11 | 3.0 | 18.9 | 20 |
| 3a | 3 | 2.1 | 6.7 | 0.15 | 3.3 | 20.8 | 22 |
| 2b | 13 | 1.4 | 7.5 | 0.15 | 3.7 | 24.2 | 66 |
| 2a | 15 | 2.8 | 6.9 | 0.14 | 3.0 | 24.0 | 62 |
| 1 | 30 | 3.6 | 5.2 | 0.01 | 0.9 | 23.5 | 59 |
| (b) 1995 | | | | | | | |
| 6 | -18 | 2.2 | 7.5 | 0.33 | 1.7 | 22.2 | 44 |
| 5 | -0.1 | 1.5 | 7.3 | 0.38 | 2.5 | 21.4 | 19 |
| 4 | 1 | 1.7 | 7.5 | 0.37 | 2.5 | 21.9 | 43 |
| 3 | 3 | 2.4 | 7.2 | 0.37 | 2.7 | 21.2 | 34 |
| 2 | 15 | 2.3 | 7.4 | 0.27 | 4.5 | 23.2 | 53 |
| 1 | 30 | 3.9 | 5.8 | 0.02 | 1.8 | 22.9 | 58 |

a Distance (dist) refers to kilometers downstream of the EB confluence